# SMART ALPR IOT: ENTRENAMIENTO Y EVALUACIÓN DE MODELO YOLO
## Detección de Placas Vehiculares Colombianas (YOLOv8n / YOLO11n)
**Proyecto Final — Visión Artificial | Universidad de La Guajira**

Este notebook está optimizado para ejecutarse en **Google Colab** con aceleración GPU (NVIDIA T4 / A100). Permite entrenar, evaluar y exportar los pesos `best.pt` y `best.onnx` utilizados por el pipeline de inferencia local del sistema Smart ALPR IoT.

### 1. Verificación de Entorno y Hardware GPU

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Dispositivo GPU: {torch.cuda.get_device_name(0)}")

### 2. Instalación de Dependencias

In [ ]:
%pip install ultralytics roboflow opencv-python matplotlib seaborn

### 3. Descarga y Preparación del Dataset de Placas Colombianas (Roboflow)
El dataset contiene imágenes etiquetadas con bounding boxes en formato YOLOv8 con clases `placa` (carros y motos).

In [ ]:
import os
from roboflow import Roboflow

# Configurar API Key de Roboflow o usar dataset público de placas
# ROBOFLOW_API_KEY = "TU_API_KEY_AQUI"
# rf = Roboflow(api_key=ROBOFLOW_API_KEY)
# project = rf.workspace("colombian-plates").project("license-plate-detection-co")
# dataset = project.version(1).download("yolov8")

# Simulación de estructura data.yaml para entrenamiento reproducible
yaml_content = """
path: ./dataset
train: images/train
val: images/val
test: images/test

names:
  0: placa
"""
os.makedirs("dataset", exist_ok=True)
with open("dataset/data.yaml", "w") as f:
    f.write(yaml_content.strip())

print("Estructura de configuración data.yaml lista.")

### 4. Configuración y Entrenamiento de YOLOv8n

In [ ]:
from ultralytics import YOLO

# Inicializar modelo preentrenado base
model = YOLO("yolov8n.pt")

# Ejecutar entrenamiento con hiperparámetros optimizados para placas viales
results = model.train(
    data="dataset/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    patience=10,
    save=True,
    device=0 if torch.cuda.is_available() else "cpu",
    workers=4,
    mosaic=1.0,
    mixup=0.1,
    degrees=10.0,
    shear=2.0,
    project="smart_alpr_runs",
    name="yolov8n_colombia_plates"
)

### 5. Métricas de Validación (mAP50, Precision, Recall y Matriz de Confusión)

In [ ]:
# Validar sobre el split de validación/test
metrics = model.val()

print(f"Precision Media: {metrics.box.mp:.4f}")
print(f"Recall Medio:    {metrics.box.mr:.4f}")
print(f"mAP@50:          {metrics.box.map50:.4f}")
print(f"mAP@50-95:       {metrics.box.map:.4f}")

### 6. Visualización de Gráficas de Rendimiento (Losses & PR Curve)

In [ ]:
import glob
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Mostrar curvas generadas por Ultralytics
plots = glob.glob("smart_alpr_runs/yolov8n_colombia_plates/*.png")
for p in plots[:4]:
    plt.figure(figsize=(10, 6))
    img = mpimg.imread(p)
    plt.imshow(img)
    plt.title(os.path.basename(p))
    plt.axis("off")
    plt.show()

### 7. Exportación a Formato ONNX y Descarga de Pesos

In [ ]:
# Exportar para aceleración en tiempo real en CPU/Edge
onnx_path = model.export(format="onnx", dynamic=False, simplify=True)
print(f"Modelo exportado exitosamente a ONNX: {onnx_path}")

# Instrucción para copiar los pesos best.pt a la carpeta models/ del proyecto local
# from google.colab import files
# files.download("smart_alpr_runs/yolov8n_colombia_plates/weights/best.pt")